<a href="https://colab.research.google.com/github/profcomff/chatbot-mark-api/blob/feature%2Fqdrant/Create_db_qdrant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !rm -rf /content/chatbot-mark-api
# !rm -rf /content/qdrant_db/

QDRANT_DIR = "./qdrant_db"

In [2]:
# !git clone https://github.com/profcomff/chatbot-mark-api.git
# !git clone --branch dev_fedor https://github.com/profcomff/chatbot-mark-api.git
!git clone --branch feature/qdrant https://github.com/profcomff/chatbot-mark-api.git

Cloning into 'chatbot-mark-api'...
remote: Enumerating objects: 520, done.
remote: Counting objects: 100% (351/351), done.
remote: Compressing objects: 100% (220/220), done.
remote: Total 520 (delta 181), reused 251 (delta 110), pack-reused 169 (from 1)
Receiving objects: 100% (520/520), 4.95 MiB | 10.05 MiB/s, done.
Resolving deltas: 100% (233/233), done.


# Библиотеки


In [7]:
!pip install langchain transformers sentence-transformers -q
!pip install -U langchain-community -q
!pip install -qU langchain-qdrant
!pip install langchain_huggingface -q

!pip install pymystem3 -q

In [8]:
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from transformers import XLMRobertaTokenizer, XLMRobertaModel
import torch

from langchain.schema import Document

from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

from langchain_core.documents import Document

import nltk
nltk.download('punkt_tab')
nltk.download('stopwords')

#для препроцессинга
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import re

from pymystem3 import Mystem

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Функции/классы

In [9]:
import sys
sys.path.append("/content/chatbot-mark-api")

from search.nn import E5LangChainEmbedder
from search.search import get_context, generate_keywords_dict, get_documents_from_qdrant

Installing mystem to /root/.local/bin/mystem from http://download.cdn.yandex.net/mystem/mystem-3.1-linux-64bit.tar.gz


In [10]:
def safe_add_documents(vector_store, chunks, batch_size=1000):
    with tqdm(total=len(chunks), desc="Добавление в БД", unit="doc") as pbar:
        for i in range(0, len(chunks), batch_size):
            try:
                batch = chunks[i:i+batch_size]
                vector_store.add_documents(batch)
                pbar.update(len(batch))
            except Exception as e:
                if "Batch size" in str(e) and "greater than max" in str(e):
                    new_size = batch_size // 2
                    print(f"Ошибка: {e}. Уменьшаю размер батча до {new_size}")
                    return safe_add_documents(vector_store, chunks[i:], new_size)
                raise
            finally:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
    print("Все документы успешно добавлены!")

# 0. Загрузка контекстов / скачивание модели


In [11]:
# answers = pd.read_excel('/content/chatbot-mark-api/file/database_v2.xlsx') #!!! should change
# answers = pd.read_excel('/content/chatbot-mark-api/file/database_v2_key_words.xlsx')
answers = pd.read_excel('/content/chatbot-mark-api/file/database_v4_key_words.xlsx')

display(answers.answer[0])
display(answers.head(2))

'Карта зачет. https://vk.com/wall-24234717_22977\nЭто ваш профсоюзный билет. С помощью этой карты вы можете получать скидки у полезных для студентов популярных брендов, участвовать в конкурсах и розыгрышах, а также посещать концерты и мероприятия. \nПолный перечень скидок есть в статье: vk.cc/bYSCNw.'

,Unnamed: 0,topic_name,answer,id,Key words,structure
0,0,Карта зачет,Карта зачет. https://vk.com/wall-24234717_2297...,0,Карта зачет,NaN
1,1,Как вступить в профсоюз,Как вступить в профсоюз? Чтобы вступить в Проф...,1,Профсоюз,NaN


link to model in HuggingFace [e5-base-en-ru](https://huggingface.co/d0rj/e5-base-en-ru)

In [12]:
tokenizer = XLMRobertaTokenizer.from_pretrained("d0rj/e5-base-en-ru", use_cache=False)
search_model = XLMRobertaModel.from_pretrained("d0rj/e5-base-en-ru", use_cache=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/471 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/702 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/529M [00:00<?, ?B/s]

# Qdrant

In [19]:
# !rm -rf /content/qdrant_db/

In [13]:
all_chunks = []

for answer, topic_name, kw in zip(answers['answer'], answers['topic_name'], answers['Key words']):
    all_chunks.append(Document(
        page_content=answer,
        metadata={
            "source": topic_name.strip(),
            "key_words": kw,
        }
    ))

embedder = E5LangChainEmbedder(
    tokenizer=tokenizer,
    model=search_model,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    add_prefix=True,
    disable_tqdm=False,
)

qdrant_client = QdrantClient(
    # Choose one of the following options:

    # Option 1: In-memory (for testing)
    # location=":memory:"

    # Option 2: Local persistence (similar to Chroma's persist_directory)
    path="./qdrant_db"  # Replace with your desired path

    # Option: Connect to a remote Qdrant instance
    # url="https://your-qdrant-url.com",
    # api_key="your-api-key"  # If authentication is required
)

qdrant_client.create_collection(
    collection_name="demo_collection",
    vectors_config=VectorParams(size=768, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name="demo_collection",
    embedding=embedder,
)

safe_add_documents(vector_store, all_chunks)

Вычисление эмбеддингов: 100%|██████████| 1/1 [00:00<00:00,  3.79batch/s]


Добавление в БД:   0%|          | 0/98 [00:00<?, ?doc/s]


Вычисление эмбеддингов: 100%|██████████| 8/8 [00:25<00:00,  3.22s/batch]

Вычисление эмбеддингов: 100%|██████████| 5/5 [00:20<00:00,  4.15s/batch]


Все документы успешно добавлены!


# Подключение Qdrant

In [14]:
# qdrant_client = QdrantClient(
#     path="./qdrant_db"  # Replace with your desired path
# )

# qdrant_client.create_collection(
#     collection_name="demo_collection",
#     vectors_config=VectorParams(size=768, distance=Distance.COSINE),
# )

vector_store = QdrantVectorStore(
    client=qdrant_client,
    collection_name="demo_collection",
    embedding=embedder,
)

Вычисление эмбеддингов: 100%|██████████| 1/1 [00:00<00:00, 15.13batch/s]


In [15]:
query = "профком?"

relevant_docs = vector_store.similarity_search(
    query,
    k=3,
)

In [16]:
relevant_docs

[Document(metadata={'source': 'Режим работы профкома', 'key_words': 'Профком', '_id': 'f8d52dc7bff3465988daa61c12ace203', '_collection_name': 'demo_collection'}, page_content='Режим работы профкома. Режим работы Профкома.  Часы работы: 11:00-16:00, ПН-ПТ. \nКабинет 2-39'),
 Document(metadata={'source': 'Связь с профкомом', 'key_words': 'Профком, группа профкома', '_id': '6afe1dfdaf5843c390e193cc6351163c', '_collection_name': 'demo_collection'}, page_content='Связь с профкомом. Как связаться с Профкомом? Вы можете написать нам в личных сообщениях в группе Профкома ВКонтакте (https://vk.com/profcomff), а также в нашем телеграмм-канале (https://t.me/profcom_ff).'),
 Document(metadata={'source': 'Начало работы в профкоме', 'key_words': 'Профком', '_id': '19023ba166d045aca82a7e0bd5b51c75', '_collection_name': 'demo_collection'}, page_content='Начало работы в профкоме. Как начать что-то делать в Профкоме? Чтобы стать активистом Профкома, можно написать в личные сообщения группы https://vk.co

In [ ]:
!zip -r qdrant_db.zip qdrant_db/

# Для проверки key_words

In [18]:
# points, next_page = vector_store.client.scroll(
#     collection_name=vector_store.collection_name,
#     # limit=100,
#     with_payload=True
# )

# while points:
#     for point in points:
#         doc_id = str(point.id)
#         payload = point.payload or {}
#         metadata = payload.get("metadata", payload) if isinstance(payload, dict) else {}

#         key_words_val = metadata.get("key_words")
#         print(key_words_val)